Gold Layer: The Gold layer is the final and most important layer in a modern data pipeline (Bronze -> Silver -> Gold). It is where raw data is transformed into business-ready datasets that are optimized for analytics, dashboards, and decision-making.

#### Creating Schema for Gold Layer

In [0]:
spark.sql("create schema if not exists workspace.gold")

DataFrame[]

#### Loading data from Silver layer

In [0]:
orders_df = spark.table('workspace.silver.orders')
customer_df = spark.table('workspace.silver.customers')
order_items_df = spark.table('workspace.silver.order_items')
products_df = spark.table('workspace.silver.products')
sellers_df = spark.table('workspace.silver.sellers')
order_review_df = spark.table('workspace.silver.order_reviews')
payment_df = spark.table('workspace.silver.order_payments')

#### Fact Tables

In [0]:
from pyspark.sql.functions import *
fact_order_items = order_items_df.join(orders_df.select('order_id','order_purchase_timestamp'), on = 'order_id', how = 'left')

fact_order_items = fact_order_items.withColumn('purchase_date_key',date_format("order_purchase_timestamp",'yyyyMMdd').cast('int'))

#fact_order_items.display()

In [0]:
fact_payment = payment_df.select('order_id','payment_sequential','payment_type','payment_installments','payment_value')

#display(fact_payment)

In [0]:
fact_reviews = order_review_df.select('review_id','order_id','review_score','review_comment_title','review_comment_message',
                                      'review_creation_date','review_answer_timestamp')
fact_reviews = fact_reviews.withColumn('review_creation_date_key',date_format("review_creation_date",'yyyyMMdd').cast('int'))
#display(fact_reviews)

Fact_orders consolidated table

In [0]:
avg_scores = order_review_df.groupBy('order_id')\
    .agg(round(avg('review_score'), 2).alias('avg_review_score'))
display(avg_scores)

payment_agg = payment_df.groupBy('order_id').agg(
    round(sum('payment_value'), 2).alias('total_payment_value')
)
display(payment_agg)

order_id,avg_review_score
73fc7af87114b39712e6da79b0a377eb,4.0
a548910a1c6147796b98fdf73dbeba33,5.0
f9e4b658b201a9f2ecdecbb34bed034b,5.0
658677c97b385a9be170737859d3511b,5.0
8e6bfb81e283fa7e4f11123a3fb894f1,5.0
b18dcdf73be66366873cd26c5724d1dc,1.0
e48aa0d2dcec3a2e87348811bcfdf22b,5.0
c31a859e34e3adac22f376954e19b39d,5.0
9c214ac970e84273583ab523dfafd09b,5.0
b9bf720beb4ab3728760088589c62129,4.0


order_id,total_payment_value
b81ef226f3fe1789b1e8b2acac839d17,99.33
a9810da82917af2d9aefd1278f1dcfa0,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,65.71
ba78997921bbcdc1373bb41e913ab953,107.78
42fdf880ba16b47b59251dd489d4441a,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,96.12
771ee386b001f06208a7419e4fc1bbd7,81.16
3d7239c394a212faae122962df514ac7,51.84
1f78449c87a54faf9e96e88ba1491fa9,341.09
0573b5e23cbd798006520e1d5b4c6714,51.95


In [0]:
fact_orders = orders_df.join(payment_agg, on='order_id', how='left')\
                        .join(avg_scores,  on='order_id', how='left')\
                     .select('order_id','customer_id','order_status','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date','delay_days','delivery_days','total_payment_value','avg_review_score')\
                     .withColumn('late_delivery',when(col('delay_days') > 0, lit('Yes')).otherwise(lit('No')))\
                     .withColumn('purchase_date_key',date_format("order_purchase_timestamp",'yyyyMMdd').cast('int'))\
                     .withColumn('delivery_date_key',date_format("order_delivered_customer_date",'yyyyMMdd').cast('int'))\
                     .withColumn('estimated_delivery_date_key',date_format("order_estimated_delivery_date",'yyyyMMdd').cast('int'))
                        

#display(fact_orders)

#### Dimension tables

In [0]:
dim_customer = customer_df.select('customer_id','customer_unique_id',
                                  'customer_zipcode','customer_city','customer_state',
                                  col('latitude').alias('customer_latitude'),
                                  col('longitude').alias('customer_longitude'))
#display(dim_customer)


    
dim_seller = sellers_df.select('seller_id',
                                  'seller_zipcode','seller_city','seller_state',
                                  col('latitude').alias('seller_latitude'),
                                  col('longitude').alias('seller_longitude'))
#display(dim_seller)

dim_product = products_df.select('product_id','product_category_name','product_weight_gm',
                                 'product_length_cm','product_height_cm','product_width_cm','product_photos_qty')
#display(dim_product)


In [0]:
dim_date = spark.sql("select explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as full_date").select(
    date_format("full_date", "yyyyMMdd").cast('int').alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    month("full_date").alias("month"),
    quarter("full_date").alias("quarter"),
    dayofmonth("full_date").alias("day_of_month"),
    dayofweek("full_date").alias("day_of_week"),     
    date_format("full_date", "EEEE").alias("day_name"),
    date_format("full_date", "MMMM").alias("month_name"),
    weekofyear("full_date").alias("week_of_year"),
    when(dayofweek("full_date").isin([1, 7]), 1)
     .otherwise(0).alias("is_weekend"),
)


In [0]:
fact_order_items.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in fact_order_items.columns]).show()

#Dropping 8 order_item rows that has no match in orders as no parent order can't link to any customer, no purchase date can't do time analysis and no order_status can't filter by delivery status
fact_order_items = fact_order_items.filter(col("order_purchase_timestamp").isNotNull())

+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|total_price|order_purchase_timestamp|purchase_date_key|
+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+
|       0|            0|         0|        0|                  0|    0|            0|          0|                       8|                8|
+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+



#### Writing to Gold (delta tables)


In [0]:
def write_to_table(df, table_name):
    full_table = f'workspace.gold.{table_name}'
    df.write.format('delta').mode('overwrite') \
        .option('overwriteSchema', 'true').saveAsTable(full_table)

    print(f'{table_name} loaded')

In [0]:
write_to_table(fact_orders,'fact_orders')
write_to_table(fact_order_items,'fact_order_items')
write_to_table(fact_reviews,'fact_reviews')
write_to_table(fact_payment,'fact_payment')

write_to_table(dim_customer,'dim_customer')
write_to_table(dim_seller,'dim_seller')
write_to_table(dim_product,'dim_product')
write_to_table(dim_date,'dim_date')

fact_orders loaded
fact_order_items loaded
fact_reviews loaded
fact_payment loaded
dim_customer loaded
dim_seller loaded
dim_product loaded
dim_date loaded


#### Writing to CSV

In [0]:
#mapping of names to be used for csv files
name_map = {
    "orders": fact_orders,
    "order_items": fact_order_items,
    "order_payments": fact_payment,
    "order_reviews": fact_reviews,
    "customers": dim_customer,
    "products": dim_product,
    "sellers": dim_seller,
    "dates": dim_date
}
base_path = "/Volumes/olist_ecommerce/default/cleaned_data/"
for file_name, df in name_map.items():
    try:
        temp_path = f"{base_path}{file_name}_tmp"
        final_path = f"{base_path}{file_name}.csv"

        df.coalesce(1).write.csv(temp_path,mode='overwrite',header=True)
        files = dbutils.fs.ls(temp_path)
        csv_file = [f.path for f in files if f.path.endswith(".csv")][0]

        dbutils.fs.cp(csv_file, final_path)
        dbutils.fs.rm(temp_path, recurse=True)
        print(f"{file_name}.csv created")

    except Exception as e:
        print(f"Failed {file_name}: {str(e)}")

orders.csv created
order_items.csv created
order_payments.csv created
order_reviews.csv created
customers.csv created
products.csv created
sellers.csv created
dates.csv created
